In [122]:
import numpy as np
import re
import polars as pl
import scipy

from langchain.document_loaders import JSONLoader
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents.base import Document


# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)



In [123]:
path = "/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json"

In [124]:
# question = "어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘"
question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"
# question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구를 한 대학 어디야?"

In [125]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            # [f"{d.metadata['seq_num']}-{d.metadata['sub_seq_num']} RANK:{i+1}\n{d.metadata['title'][:20]}:\n\n" + d.page_content for i, d in enumerate(docs)]
            [f"{d.metadata['seq_num']} RANK:{i+1}\n{d.metadata['title'][:64]}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["title"] = record.get("title")
    metadata["date"] = record.get("date")
    return metadata

loader = JSONLoader(
    file_path=path,
    jq_schema=".[]",
    content_key="content",
    text_content=True,
    metadata_func=metadata_func
)
text_splitter = RecursiveCharacterTextSplitter(
    separators="\n\n",
    chunk_size=200,
    chunk_overlap=0,
    keep_separator=True
)
embeddings = OllamaEmbeddings(
    model="gemma-2-embed"
)

# Test Embedding of LangChain

In [126]:
r1 = embeddings.embed_documents(
    [
        "Alpha is the first letter of Greek alphabet",
        "Beta is the second letter of Greek alphabet",
    ]
)

# Load Vector DB
> Load and Split JSON files by JSONLoader of LangChain

In [127]:
df = pl.read_json(path)

In [128]:
df.head()

title,content,date
str,str,str
"""미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착""","""미세플라스틱(Microplastic)이 환경에서 노후화…","""2022-04-09"""
"""UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스…","""유엔환경계획(UNEP), 우려되는 화학물질 및 고분자(…","""2023-09-16"""
"""APEC 국가간 무역장벽, 태평양 해양쓰레기 청소 방해""","""태평양의 플라스틱 등 해양쓰레기 청소가 국가 간 무역장…","""2022-12-18"""
"""생수는 과연 안전할까?···건강 위협하는 미세 플라스틱""","""플라스틱병에 든 생수에 미세플라스틱이 다량 함유되어있다…","""2024-07-31"""


In [129]:
data = loader.load()

In [130]:
data

[Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content='미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.\n\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(methylene blue)에 노출시켰다. 표면의 풍화는 표면적 및 표면화학 성질을 변화시켜 오염물질의 흡착을 증가시키는 결과를 나타냈다.\n\n연구팀의 모델 시스템은 미세플라스틱 노후화의 초기 단계를 보여주지만, 실제 미세플라스틱은 환경에서 수 십 년 동안 풍화되므로, 실험조건보다 더욱 노화된 미세플라스틱이 실제 환경에서 유기오염물질과의 상호작용을 통해 더욱 악화된 환경위해를 초래할 수 있는지에 대한 시사점을 제시하고 있다.\n\n연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.\n\nPerreault 박사는 ‘이번 연구는 환경 내에서의 풍화작용이 미세플라스틱의 거동에 어떤 영향을 미치

In [131]:
splits = text_splitter.split_documents(data)

In [132]:
splits

[Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content='미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.'),
 Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content='\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(methylene blue)에 노출시켰다. 표면의 풍화는 표면적 및 표면화학 성질을 변화시켜 오염물질의 흡착을 증가시키는 결과를 나타냈다.'),
 Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content=

In [133]:
vectordb = FAISS.from_documents(documents=splits, embedding=embeddings)

In [134]:
vectordb.docstore.__dict__

{'_dict': {'821872c2-64bb-40aa-9230-b02ec053aeda': Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content='미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.'),
  'de8469d0-1ba6-48bf-bbef-61ce15aa3b6c': Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'seq_num': 1, 'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09'}, page_content='\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(methylene blue)에 노출시켰다. 표면의 풍화는 표면적 및 표면화학 성질을 변화시켜 오염물질의 흡착을 증가시키는 결과를 나타냈다.'),
  'eb44c2d8-8125-43e2-ab1e-ed78a45ed988': Document(metadata={'source': '/mnt/d/temp/user/ed/mart/mmpl

# Invoke Ollama

- https://python.langchain.com/v0.2/docs/integrations/chat/ollama/

In [135]:
llm = ChatOllama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
    num_gpu=-1
)

response = llm.invoke(question)
print(f"[{response.response_metadata['eval_duration'] / np.power(10., 9)} sec.]:\n" + "" + response.content)


[7.009067 sec.]:
죄송하지만, 해당 연구와 관련된 미국 대학교에 대한 정보는 가지고 있지 않습니다. 미세플라스틱과 같은 환경 문제에 관심이 많으시군요! 햇빛에 노출되면 오염줄질을 흡수하는 미세플라스틱의 특징은 매우 중요하며, 이에 대한 연구를 수행한 미국 대학교가 있을 가능성도 있습니다.

해당 정보를 찾기 위해, 관련 학술잡지나 논문 데이터베이스에서 "미세플라스틱"과 "햇빛", 그리고 "오염줄질" 등의 키워드로 검색하면 도움이 될 수 있습니다. 또한, 미세플라스틱 연구에 전문적인 미국 대학교 목록을 찾아보고 해당 연구를 수행했는지 확인해 볼 수도 있습니다.

미세플라스틱 문제에 대한 더 많은 정보를 얻으시려면 다음과 같은 웹사이트에서 검색해 보세요.

* National Oceanic and Atmospheric Administration (NOAA)
* Environmental Protection Agency (EPA)
* United States Geological Survey (USGS)



# Multi Query Retriever

In [136]:
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm
)

In [137]:
docs = mq_retriever.invoke(question)
len(docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Which American universities have conducted research on the ability of microplastics to absorb pollutants when exposed to sunlight?', '2. What are some US colleges or universities that have performed studies on the interaction between microplastic exposure to UV radiation and pollutant absorption?', '3. Can you find any American universities involved in research exploring the relationship between sunlight exposure, microplastic degradation, and pollutant uptake by these particles?']


5

## Alternative to MultiQueryRetriever

## Keyword Extraction

In [138]:
keyword_exxtraction_prompt = PromptTemplate(
    input_variables=["question"],
    template="""아래 질문에서 누가, 무엇을, 어떻게, 이유 등과 관련된 가장 중요한 키워드만 숫자없이 답변해줘
    기본 질문: {question}
    """
)
keyword_exxtraction_chain = LLMChain(llm=llm, prompt=keyword_exxtraction_prompt)

In [139]:
keywords_response = keyword_exxtraction_chain.invoke(question)
print(f"{keywords_response['question']}\n{'-'*64}\n{keywords_response['text']}")

미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어
----------------------------------------------------------------
미세플라스틱, 햇빛, 오염줄질, 미국 대학교.


## Query Expansion

In [140]:
query_expansion_prompt = PromptTemplate(
    input_variables=["question"],
    template=""" 기본 질문의 동사나 용언 형태를 바꾼 3가지 다른 질문들을 만들어줘.
    - 질문은 \n로 구분해서 나열
    - 원래 질문과 비슷한 길이로 3가지 질문만 짧게 나열
    - 질문앞에 기호 절대 붙이지 마
    - 질문에 순서나 번호는 붙이지 마. 한글로만 답변
    기본 질문: {question}
    """
)
query_expansion_chain = LLMChain(llm=llm, prompt=query_expansion_prompt)

In [141]:
queries_response = query_expansion_chain.invoke(question)
print(f"{queries_response['question']}\n{'-'*64}\n{queries_response['text']}")

미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어
----------------------------------------------------------------
1 미세플라스틱의 햇빛 노출과 오염줄질 흡수 현상에 대한 미국 대학 연구는 어느 곳에서 이루어지나요?
2 미세플라스틱이 햇빛에 노출되었을 때 오염 줄질을 얼마나 흡수하는가에 관한 연구를 수행한 미국 대학교가 있다면 알려주세요.
3 미국 대학의 미세플라스틱과 햇빛, 그리고 오염줄질 관련된 연구 결과를 보여줘


In [142]:
chain_response = queries_response

In [143]:
expansions = list(filter(lambda s: s, chain_response['text'].split('\n')))
expansions

['1 미세플라스틱의 햇빛 노출과 오염줄질 흡수 현상에 대한 미국 대학 연구는 어느 곳에서 이루어지나요?',
 '2 미세플라스틱이 햇빛에 노출되었을 때 오염 줄질을 얼마나 흡수하는가에 관한 연구를 수행한 미국 대학교가 있다면 알려주세요.',
 '3 미국 대학의 미세플라스틱과 햇빛, 그리고 오염줄질 관련된 연구 결과를 보여줘']

In [144]:
expanded_queries = [Document(page_content=e) for e in expansions]
expanded_queries

[Document(page_content='1 미세플라스틱의 햇빛 노출과 오염줄질 흡수 현상에 대한 미국 대학 연구는 어느 곳에서 이루어지나요?'),
 Document(page_content='2 미세플라스틱이 햇빛에 노출되었을 때 오염 줄질을 얼마나 흡수하는가에 관한 연구를 수행한 미국 대학교가 있다면 알려주세요.'),
 Document(page_content='3 미국 대학의 미세플라스틱과 햇빛, 그리고 오염줄질 관련된 연구 결과를 보여줘')]

## Retrieve with the expanded queries

In [145]:
top_k_retrieval = 2

In [146]:
retriever = vectordb.as_retriever(
    # search_type="similarity_score_threshold",
    search_kwargs={
        "k": top_k_retrieval,
        # "score_threshold": 0.5
    }
)
# retriever = FAISS.from_documents(splits, embeddings).as_retriever()

In [147]:
total_docs = []

docs = retriever.invoke(question)
total_docs.extend(docs[:top_k_retrieval])

pretty_print_docs(docs)

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질


In [148]:
for q in expanded_queries:
    docs = retriever.invoke(q.page_content)
    for doc in docs[:top_k_retrieval]:
        if doc not in total_docs:
            total_docs.append(doc)
pretty_print_docs(total_docs)

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질
----------------------------------------------------------------------------------------------------
2 RANK:3
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 방안 3 : 가장 구속력이 낮은 방안으로, 화학물질 첨가 및 사용에 대하여 각 국가의 재량에 맡기고 개발된 전략을 공유하도록 함

 

우려되는 화학물질 및 고분자 정의

조약의 효력은 우려되는 화학물질 및 고분자의 정의에 따라 달라질 수 있으며, 다음과 같은 사항이 포함됨.
----------------------------------------------------------------------------------------------------
4 RANK:4
생수는 과연 안전할까?···건강 위협하는

## Rerank the retrieved results

In [149]:
import numpy as np
from typing import List, Tuple
from langchain_community.cross_encoders import BaseCrossEncoder

class OllamaCrossEncoder(BaseCrossEncoder):
    def __init__(self, embeddings: OllamaEmbeddings):
        self.embeddings = embeddings

    def score(self, text_pairs: List[Tuple[str, str]]) -> List[float]:
        query_embedding = self.embeddings.embed_query(text_pairs[0][0])
        doc_embeddings = self.embeddings.embed_documents([p[1] for p in text_pairs])
        # TODO cossine sim calc of NNC of tweak
        scores = []
        for doc_embed in doc_embeddings:
            scores.append(1 - scipy.spatial.distance.cosine(np.array(query_embedding), np.array(doc_embed)))
        return scores


In [150]:
cross_encoder = OllamaCrossEncoder(embeddings=embeddings)

In [151]:
text_pairs = [(question, d.page_content) for d in total_docs]
scores = cross_encoder.score(text_pairs)
print(scores)


[0.8892599042528533, 0.8841447672937507, 0.826115659403217, 0.8565824095158067]


In [152]:
top_k_doc_indexes = np.argsort(scores)[::-1][:2].tolist()
top_k_doc_indexes

[0, 1]

In [153]:
pretty_print_docs([total_docs[i] for i in top_k_doc_indexes])

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질


In [182]:
ranking_response = llm.invoke(f"""
아래 문장을 참고해서 질문에 맞는 답만 짧게 답변해줘. 문장 내용만 참고해서 답해줘. 한글로만 답변해
{total_docs[0]}
{total_docs[1]}
질문: {question}
""")
ranking_response.content

ResponseError: tiger-gemma2 does not support tools

# Multi query retriever
> alternative to MultiQueryRetriever
>> CANNOT use custom logics to postprocess retrieved docs
>> BUG: CANNOT use include_original=True if llm is local model.

In [155]:
llm_chain = keyword_exxtraction_chain

In [156]:
mq_retriever = MultiQueryRetriever(
    k=3,
    include_original=True,
    retriever=vectordb.as_retriever(),
    llm_chain=llm_chain,
    verbose=True,
    parser_key="lines",
)

In [157]:
import pytest

with pytest.raises(AttributeError, match="""'str' object has no attribute 'append'"""):
    docs = mq_retriever.invoke(question)
    len(docs)

INFO:langchain.retrievers.multi_query:Generated queries: 미세플라스틱, 햇빛, 오염 줄기(적은 용량의 물/화학약제로 부패 또는 소독), 미국 대학교.


In [158]:
mq_retriever = MultiQueryRetriever(
    k=3,
    include_original=False,
    retriever=vectordb.as_retriever(),
    llm_chain=llm_chain,
    verbose=True,
    parser_key="lines",
)

In [159]:
docs = mq_retriever.invoke(question)
assert len(docs) > 0
print(question)
pretty_print_docs(docs)

INFO:langchain.retrievers.multi_query:Generated queries: 미세플라스틱, 햇빛, 오염줄질, 연구, 미국 대학교


미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어
4 RANK:1
생수는 과연 안전할까?···건강 위협하는 미세 플라스틱:


 먹는물네트워크와 대한환경공학회는 31일 서울 중구 환경재단 레이첼카슨홀에서 ‘생수와 미세플라스틱, 안전한 먹는 물을 위한 공동 노력’ 포럼을 열고 이같이 지적했다. 이날 발제자로 나선 안윤주 건국대 환경보건과학과 교수는 생수에서 미세플라스틱이 다량으로 검출됐다는 연구 결과들에 대해 “입자 분석기술의 발달로 나노 크기의 플라스틱 조각 검출이 가능해지면서 (제대로 된) 결과가 나오고 있다”고 설명했다. 
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

조약에 명시된 요구사항은 국가 정책기관에 전달되어 궁극적으로 플라스틱을 생산 및 사용하는 기업과, 화학첨가제를 제조 및 공급하는 기업에 영향을 미칠 예정.
----------------------------------------------------------------------------------------------------
2 RANK:3
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

조약 초안에 따르면, 조약 당사국은 미세플라스틱을 포함한 유해화학물질이 대기, 토양, 수질 및 생태계로 배출되는 것을 ‘방지 및 제거‘ 해야 한다고 명시하고 있으며, 이를 위해 아래 3 가지 방안을 제시함.
- 방안 1 : 각 국가의 플라스틱 생산에 우려되는 화학물질 및 고분자 사용을 ‘금지 및 제거‘하도록 구속함
----------------------------------------------------------------------------------------------------
2 RANK:4

# Contextual compression

## LLM Chain Filter

In [160]:
keywords_response["text"]

'미세플라스틱, 햇빛, 오염줄질, 미국 대학교.'

In [161]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



# prompt_template = PromptTemplate(
#     input_variables=["filter_prompt", "question", "context"],
#     template="아래 키워드 모두와 관련이 있는 문서면 1을, 아니면 0을 답변해줘. 답변만 짧게해줘\n {filter_prompt}",
# )
# llm_chain = LLMChain(llm=llm, prompt=prompt_template, output_parser=StrOutputParser())
# llm_filter = LLMChainFilter(llm_chain=llm_chain)

prompt_template = PromptTemplate.from_template(
    "아래 키워드 모두와 관련이 있는 문서면 1을, 아니면 0을 답변해줘. 답변만 짧게해줘\n {filter_prompt}",
    partial_variables={"filter_prompt": keywords_response["text"]}
)
# prompt = prompt_template.format(filter_prompt=keywords_response["text"])
# llm_filter = LLMChainFilter.from_llm(llm, prompt=prompt_template, output_parser=StrOutputParser())

llm_filter = LLMChainFilter.from_llm(llm)


compression_retriever = ContextualCompressionRetriever(
    base_compressor=llm_filter, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    # prompt_template
    question
    # expanded_queries[0].page_content,

    # [
    #     ('question', expanded_queries[0].page_content),
    #     ("filter_prompt", keywords_response["text"])
    # ]
)
pretty_print_docs(compressed_docs)



## LLMChainExtractor

In [162]:
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    question
)
pretty_print_docs(compressed_docs)

## LLMListwiseRerank

In [163]:
from langchain.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

reranker = LLMListwiseRerank.from_llm(llm, top_n=1)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=retriever
)

try:
    compressed_docs = compression_retriever.invoke(
        question
    )
    pretty_print_docs(compressed_docs)
except Exception as e:
    logging.error(str(e))
    # ResponseError: tiger-gemma2 does not support tools


ERROR:root:tiger-gemma2 does not support tools


## EmbeddingsFilter

In [164]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    question
)
pretty_print_docs(compressed_docs)

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질


## DocumentCompressorPipeline
- with EmbeddingsRedundantFilter, EmbeddingsFilter

In [165]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20, separator=". ")
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter]
)

In [166]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    question
)
pretty_print_docs(compressed_docs)

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질


## CrossEncoderReranker Compressor

In [167]:
# from langchain_core.documents.compressor import BaseDocumentCompressor
from langchain.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker

In [168]:
cross_encoder = OllamaCrossEncoder(embeddings=embeddings)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

In [169]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    question
)
pretty_print_docs(compressed_docs)

1 RANK:1
미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착:

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
----------------------------------------------------------------------------------------------------
2 RANK:2
UN, 플라스틱 내 우려물질 사용 금지를 위한 ‘플라스틱 조약‘ 초안 발표:

- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질


## TBD LLM Reranker (Retrieval + LLM Rerank)
- https://python.langchain.com/v0.2/docs/integrations/document_transformers/rankllm-reranker/

In [170]:
prompt = f"{question} 정답만 짧게 대답해\n{' '.join([d.page_content for d in compressed_docs])}"
response = llm.invoke(prompt)
response

AIMessage(content='오염물질을 흡수하는 데 사용된 고분자 재료를 포함하여, 미국 대학의 연구팀이 발표한 논문은 미세플라스틱의 광화합성에 대한 결과를 보여줍니다. 이는 태양광에 의해 발생하는 자외선과 파장으로 인해 물질의 분위기가 변하게 되는 원리를 설명합니다.', response_metadata={'model': 'tiger-gemma2', 'created_at': '2024-08-06T10:41:44.982712541Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 2831275603, 'load_duration': 31711889, 'prompt_eval_count': 310, 'prompt_eval_duration': 107641000, 'eval_count': 90, 'eval_duration': 2690838000}, id='run-c32d0547-e13f-4578-9676-d63181aa4559-0', usage_metadata={'input_tokens': 310, 'output_tokens': 90, 'total_tokens': 400})

## Multivector Retriever

### Hypothetical Queries

In [171]:
compressed_docs[1].page_content

'- 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질\n\n- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질\n\n- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질'

In [172]:
total_docs[0].page_content

'연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.'

In [173]:
type(llm)

langchain_ollama.chat_models.ChatOllama

In [174]:
from langchain_core.pydantic_v1 import BaseModel, Field


class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


llm_with_tools = llm.bind_tools([GetWeather])

In [175]:
try:
    llm_with_tools.invoke("what is the weather like in San Francisco")
except Exception as re:
    logging.error(str(re))

ERROR:root:tiger-gemma2 does not support tools


In [176]:
from typing import List
from typing_extensions import TypedDict

from langchain_anthropic import ChatAnthropic

class Address(TypedDict):
    street: str
    city: str
    state: str

def validate_user(user_id: int, addresses: List[Address]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id: (int) the user ID.
        addresses: Previous addresses.
    """
    return True

# llm = ChatAnthropic(
#     model="claude-3-sonnet-20240229"
# ).bind_tools([validate_user])
try:
    llm = llm.bind_tools([validate_user])

    result = llm.invoke(
        "Could you validate user 123? They previously lived at "
        "123 Fake St in Boston MA and 234 Pretend Boulevard in "
        "Houston TX."
    )
    print(result)
    print(result.tool_calls)
except Exception as re:
    logging.error(str(re))

ERROR:root:tiger-gemma2 does not support tools


In [177]:
functions = [
    {
        "name": "hypothetical_questions",
        "description": "Generate hypothetical questions",
        "parameters": {
            "type": "object",
            "properties": {
                "questions": {
                    "type": "array",
                    "items": {"type": "string"},
                },
            },
            "required": ["questions"],
        },
    }
]

In [178]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser


try:
    chain = (
        {"doc": lambda x: x.page_content}
        # Only asking for 3 hypothetical questions, but this could be adjusted
        | PromptTemplate.from_template(
            "Generate a list of exactly 3 hypothetical questions that the below document could be used to answer, 한글로만 답변해:\n\n{doc}"
        )
        | llm.bind_functions(
            functions=functions, function_call={"name": "hypothetical_questions"}
        )
        | JsonKeyOutputFunctionsParser(key_name="questions")
    )
except AttributeError as ae:
    logging.error(str(ae))

ERROR:root:'ChatOllama' object has no attribute 'bind_functions'


In [179]:
from langchain.tools import DuckDuckGoSearchRun
from langchain_core.output_parsers import PydanticToolsParser
from langchain_core.pydantic_v1 import BaseModel, Field


tools = [DuckDuckGoSearchRun(max_results=4, verbose=True)]

chain = (
    # {"doc": lambda x: x.page_content}
    {"doc": lambda x: x}
    # Only asking for 3 hypothetical questions, but this could be adjusted
    | PromptTemplate.from_template(
        "Generate a list of exactly 3 hypothetical questions that the below document could be used to answer, 한글로만 답변해:\n\n{doc}"
    )
    | llm.bind_tools(tools)
    # | PydanticToolsParser(tools=[])
)

In [180]:
try:
    chain.invoke(total_docs[0].page_content)
except Exception as re:
    logging.error(str(re))

ERROR:root:tiger-gemma2 does not support tools
